# hECA v2.0 -- Pooled scATAC-seq Across Five Organs

Source: Chen et al. 2025, Scientific Data. Zenodo record 15627886 (https://zenodo.org/records/15627886).

Five organs from the hECA v2.0 ATAC collection are pooled here to exceed 500,000 cells: Lung, Brain, Kidney, Heart, and Thymus. Cell types are harmonized by uHAF; cells the source could not classify (Unclassified) are dropped. The reference label used throughout this notebook is organ, which is unambiguous.

The pooled set spans several study_id values, so a pooled clustering may partly recover study of origin rather than biology.

---

## 0. Imports & Configuration

Runtime note: at SCALE = "publication" this notebook is expensive -- roughly a week of continuous compute at the n_jobs=1 it ships with. Measured order of magnitude: about 24 hours for the k-based CARVE fit (Section 1), 61 hours for the Leiden resolution fit (Section 2), 69 hours for the scaling ladder (Section 3), and 5.6 hours for the CVI sweep (Section 4). The RandomForest classifier CARVE fits per resample, not the clustering itself, accounts for roughly 90 percent of that. Parallelizing is not a free win here: the default RandomForest already sets n_jobs=-1, so raising this notebook's own n_jobs would oversubscribe on top of that. The dev scale exists for fast iteration; its runtime is not informative about publication scale.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from benchmarks._studies import (
    STUDIES,
    carve_cache_path,
    cvi_sweep,
    fit_or_load_carve,
    load_study,
    study_model_grids,
    study_resolution_grids,
    study_scaling_sweep,
)
from benchmarks.figures import (
    figure_heca_results,
    figure_reference_scatter,
    figure_study_scaling,
    prepare_composite,
)
from benchmarks.figures._heca_results import (
    AXIS_LABELS,
    DEFAULT_SCATTER_SUBSAMPLE,
    MARKER_SIZE,
)
from benchmarks.figures._paths import CASE_STUDY_DIR

RANDOM_SEED = 42
SCALE = "dev"  # switch to "publication" for the manuscript run

# Scale-qualified output, so a development figure can never be mistaken for
# one that belongs in the manuscript.
OUT_DIR = CASE_STUDY_DIR if SCALE == "publication" else CASE_STUDY_DIR / SCALE
OUT_DIR.mkdir(parents=True, exist_ok=True)

study = STUDIES["heca"]
model_grids = study_model_grids(study)
X, y, meta = load_study(study, scale=SCALE)
print(meta["n_cells"], "cells across", meta["organs"], "|", meta["scale"])


def show(fig):
    display(fig)
    plt.close(fig)

### 0.1 Reference Labels

UMAP of the 50 principal components the loader returns, colored by organ, the reference label CARVE is scored against. The source ships no coordinates, so the embedding is computed here, the way the source publication visualizes its data. A seeded UMAP runs single-threaded: about a minute at the development scale and tens of minutes at publication scale, which is negligible beside the fits below. At publication scale the scatter draws a fixed subsample of cells, the same size the composite figure's scatter panels use. The composite in Section 4 draws the same embedding, with the same colors.

In [ ]:
import umap

reference_embedding = umap.UMAP(random_state=RANDOM_SEED).fit_transform(X)
show(
    figure_reference_scatter(
        reference_embedding, y.to_numpy(),
        axis_labels=AXIS_LABELS, title="Reported Organ",
        max_points=DEFAULT_SCATTER_SUBSAMPLE, marker_size=MARKER_SIZE,
        out_dir=OUT_DIR, save_name="heca_reference_scatter.png",
    )
)

---

## 1. CARVE Analysis

CARVE is fit once at case-study scale using consensus_anchors=study.consensus_anchors, so the consensus matrix stays an anchor-by-anchor block rather than the full n-by-n matrix, which does not fit at this scale.

In [ ]:
carve = fit_or_load_carve(
    X, y,
    cache_path=carve_cache_path(
        study, scale=SCALE, root=Path("./carve_state_saves")
    ),
    model_grids=model_grids,
    random_state=RANDOM_SEED,
    consensus_anchors=study.consensus_anchors,
)

---

## 2. Leiden Resolution Sweep

SweepSpec sweeps exactly one parameter, so a k-based run and a resolution-based run cannot share one CARVE object. This is a second, independent fit over the same anchored consensus path.

In [ ]:
from carve import CARVE

leiden = CARVE(
    estimator_param_grids=study_resolution_grids(study),
    n_resamples=100,
    n_jobs=1,
    random_state=RANDOM_SEED,
    consensus_anchors=study.consensus_anchors,
).fit(X)
leiden.estimator_results_[["config_id", "resolution", "ari_stability"]]

---

## 3. Scaling Ladder

Runtime, peak memory, and the selected k are measured at a ladder of subsample sizes drawn from the same pooled data, so the underlying biology is held fixed and n is the only thing varying between rungs. The development ladder stops at 25,000 cells; the publication ladder runs the full set of sizes up to every cell in the pooled dataset.

The 500,000 rung is a nominal target, not a verified count: the pooled total after the Unclassified drop is only known once the data is loaded. A rung larger than what is actually available is skipped with a warning rather than raised as an error, so an overestimate here costs a warning, not the hours of compute already spent on the smaller rungs.

In [ ]:
LADDER = {
    "dev": [5_000, 10_000, 25_000],
    "publication": [10_000, 25_000, 50_000, 100_000, 250_000, 500_000, X.shape[0]],
}[SCALE]

sweep_df = study_scaling_sweep(
    X, y, sizes=LADDER, model_grids=model_grids,
    consensus_anchors=study.consensus_anchors, random_state=RANDOM_SEED,
)
show(figure_study_scaling(sweep_df, out_dir=OUT_DIR))
sweep_df

---

## 4. Quantitative Comparison

### 4.1 Composite Paper Figure (with ARI Comparison)

In [ ]:
curves_df, best_df = cvi_sweep(
    X, y, model_grids=model_grids, candidate_k=study.candidate_k,
    random_state=RANDOM_SEED, n_jobs=-1,
)
inputs = prepare_composite(
    X, y.to_numpy(), carve, curves_df=curves_df, best_df=best_df,
    comparison_metric="silhouette", embedding=reference_embedding,
    measure="stability", rule="1se", random_state=RANDOM_SEED,
)
show(figure_heca_results(inputs, out_dir=OUT_DIR))

---

## 5. Summary

This case study exercises CARVE at the scale a reviewer explicitly asked about: hundreds of thousands of pooled scATAC-seq cells, well past where spectral or Ward agglomerative clustering, or an exact consensus matrix, remain options. The k-based and Leiden resolution runs above are two independent CARVE fits, compared by how well each one's selected clustering agrees with the reported organ labels and with each other. The scaling ladder quantifies what that scale costs directly, in runtime and peak memory, at the largest n reached.

---